# 章节实践

## 概述

本节是本章的综合编程实践。你需要独立开发一个**带Bias的MIX模式Matmul算子**，综合运用前面学到的MIX模式架构、多核偏移计算、Bias处理和高阶API调用流程。

### 实践任务

开发一个MIX模式的Matmul算子，支持Bias加法：**C = A × B + Bias**

### 算子规格

| 参数 | name | shape | data type | format | isTrans |
| --- | --- | --- | --- | --- | --- |
| 输入 | a | [256, 256] | float16 | ND | false |
| 输入 | b | [256, 512] | float16 | ND | false |
| 输入 | bias | [1, 512] | float32 | ND | - |
| 输出 | c | [256, 512] | float32 | ND | - |

### 要求

1. 使用**MIX模式**（不加`matmul_cube_only=True`）
2. 支持**Bias**加法（`enable_bias(True)` + `set_bias`）
3. 多核并行，核数 `USE_CORE_NUM = 48`，launch 核数 `// 2`
4. 使用 `torch.allclose` 验证结果正确性
5. 核函数命名为 `matmul_mix_bias_kernel`

本实践综合了04.04（Bias处理和MIX多核偏移）的知识点，任务结构如下图所示：

<img src="./images/04.05_chapter_practice/practice_structure.png" alt="实践任务结构" width="600px">

### 开发步骤

| 步骤 | 内容 | 参考小节 |
| --- | --- | --- |
| 1 | 编写 `calc_offsets` 函数（含Bias偏移，返回6值） | 04.04 |
| 2 | 编写核函数 `matmul_mix_bias_kernel` | 04.04 |
| 3 | 编写 `generate_tiling` 函数（启用Bias） | 04.04 |
| 4 | 编写 host 调用函数 | 04.04 |
| 5 | 运行验证 | - |

### 提示

- **MIX模式**核函数不需要 `ascend_is_aic()` 守卫（参考04.04 MIX模式部分）
- **Bias偏移**：`offset_bias = n_index * tiling.single_core_n`（参考04.04 Cube Only部分）
- **set_tail**：MIX模式传入2个参数 `(tail_m, tail_n)`，tail_k使用默认值-1（参考04.04 MIX模式部分）
- **calc_offsets**：返回6值 `(offset_a, offset_b, offset_c, offset_bias, tail_m, tail_n)`，尾块判断需含 `<= 0` 条件（参考04.04 Cube Only部分）
- **Tiling**：`enable_bias(True)` + `set_bias_type(...)`（参考04.04）

In [ ]:
# 环境初始化
!mkdir -p Sources/04.05

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("环境初始化完成")

### TODO 模板

下方代码是待补齐的TODO模板。请根据提示在标记为 `# TODO` 的位置填写代码，完成带Bias的MIX模式Matmul算子。

In [ ]:
%%writefile Sources/04.05/matmul_mix_bias.py
# Copyright (c) 2025 Huawei Technologies Co., Ltd.
# CANN Open Software License Agreement Version 2.0

from typing import Tuple
import logging
import argparse
import torch

try:
    import torch_npu
except ModuleNotFoundError:
    pass

import asc
import asc.runtime.config as config
import asc.lib.runtime as rt
import asc.lib.host as host

logging.basicConfig(level=logging.INFO)

USE_CORE_NUM = 48
M_DIM = 256
N_DIM = 512
K_DIM = 256


# ========== 步骤1：calc_offsets函数 ==========
@asc.jit
def calc_offsets(tiling: asc.adv.TCubeTiling) -> Tuple[int, int, int, int, int, int]:
    """计算多核偏移，返回 (offset_a, offset_b, offset_c, offset_bias, tail_m, tail_n)"""
    block_idx = asc.get_block_idx()
    m_single_blocks = tiling.m.ceildiv(tiling.single_core_m)
    m_index = block_idx % m_single_blocks
    n_index = block_idx // m_single_blocks

    # TODO: 计算 offset_a, offset_b, offset_c, offset_bias
    offset_a = 0  # TODO: 替换为正确计算
    offset_b = 0  # TODO: 替换为正确计算
    offset_c = 0  # TODO: 替换为正确计算
    offset_bias = 0  # TODO: 替换为正确计算

    # TODO: 尾块处理（tail_m, tail_n），注意包含 <= 0 条件
    tail_m = 0  # TODO: 替换为正确计算
    tail_n = 0  # TODO: 替换为正确计算

    return offset_a, offset_b, offset_c, offset_bias, tail_m, tail_n


# ========== 步骤2：核函数 ==========
@asc.jit(always_compile=True)
def matmul_mix_bias_kernel(a: asc.GlobalAddress, b: asc.GlobalAddress, c: asc.GlobalAddress,
                            bias: asc.GlobalAddress, tiling: asc.adv.TCubeTiling,
                            workspace: asc.GlobalAddress):
    # TODO: 调用 calc_offsets 获取偏移
    pass  # TODO: 替换为正确代码

    # TODO: 创建 GlobalTensor 并绑定GM地址（带偏移）
    pass  # TODO: 替换为正确代码

    # TODO: 创建 TPipe 和 Matmul 对象（含bias类型）
    pass  # TODO: 替换为正确代码

    # TODO: register_matmul 初始化
    pass  # TODO: 替换为正确代码

    # TODO: 执行计算（set_tensor_a/b, set_bias, set_tail, iterate_all, end）
    pass  # TODO: 替换为正确代码

    asc.pipe_barrier(asc.PipeID.PIPE_ALL)


# ========== 步骤3：Tiling生成 ==========
def generate_tiling(m, n, k) -> asc.adv.TCubeTiling:
    matmul_tiling = host.MultiCoreMatmulTiling(host.get_ascendc_platform())

    # TODO: 设置 A/B/C/Bias 的类型
    pass  # TODO: 替换为正确代码

    # TODO: 设置核数、形状、启用Bias、缓冲区
    pass  # TODO: 替换为正确代码

    tiling = asc.adv.TCubeTiling()
    matmul_tiling.get_tiling(tiling)
    return tiling


# ========== 步骤4：Host调用 ==========
def matmul_mix_bias_custom(backend: config.Backend, platform: config.Platform):
    config.set_platform(backend, platform)
    device = "npu" if config.Backend(backend) == config.Backend.NPU else "cpu"

    a = torch.randint(-5, 5, (M_DIM, K_DIM), device=device).to(torch.float16)
    b = torch.randint(-5, 5, (K_DIM, N_DIM), device=device).to(torch.float16)
    bias = torch.randint(-5, 5, (1, N_DIM), device=device).to(torch.float32)
    c = torch.zeros((M_DIM, N_DIM), dtype=torch.float32, device=device)

    # TODO: 生成Tiling、创建workspace、启动核函数
    pass  # TODO: 替换为正确代码

    golden = (torch.matmul(a.to(torch.float32), b.to(torch.float32)) + bias).to(torch.float32)
    assert torch.allclose(c, golden, rtol=1e-3, atol=1e-3)
    logging.info(f"[INFO] MIX+Bias验证通过! M={M_DIM}, N={N_DIM}, K={K_DIM}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("-r", type=str, default="NPU", help="backend to run")
    parser.add_argument("-v", type=str, default=None, help="platform to run")
    args = parser.parse_args()
    backend = args.r
    platform = args.v
    if backend not in config.Backend.__members__:
        raise ValueError(f"Unsupported Backend! Supported: {list(config.Backend.__members__.keys())}")
    backend = config.Backend(backend)
    if platform is not None:
        platform_values = [p.value for p in config.Platform]
        if platform not in platform_values:
            raise ValueError(f"Unsupported Platform! Supported: {platform_values}")
        platform = config.Platform(platform)
    logging.info("[INFO] start process sample matmul_mix_bias.")
    matmul_mix_bias_custom(backend, platform)
    logging.info("[INFO] Sample matmul_mix_bias run success.")

### 运行验证

完成TODO代码后，运行下方命令验证。如果代码正确，将输出 `MIX+Bias验证通过` 和 `Sample matmul_mix_bias run success`。

> 如果遇到错误，请对照04.04（Cube Only的Bias处理和MIX模式的多核偏移）检查代码。

In [ ]:
# 运行你的实现（NPU模式）
!python3 Sources/04.05/matmul_mix_bias.py -r NPU

### 参考答案

如果你在实现过程中遇到困难，可以查看下方参考答案。建议先独立尝试，再对照答案修正。

In [ ]:
!cat ./answer/04.05_chapter_practice/matmul_mix_bias.py